In [1]:
from vizpro import CustomWidget, Datasets
import traitlets

In [3]:
affairsData = Datasets.get_dataset("affairs")
affairsData.columns

['rownames',
 'affairs',
 'gender',
 'age',
 'yearsmarried',
 'children',
 'religiousness',
 'education',
 'occupation',
 'rating']

In [9]:
jsPath = "../samples/scatterplot.js"
cssPath = "../samples/scatterplot.css"


class CustomScatterplot(CustomWidget):
    _esm = CustomWidget.createWidgetFromLocalFile(paramList=["data", "x", "y", "hue"], filePath=jsPath)
    _css = cssPath
    
    data = traitlets.List([]).tag(sync=True)
    x = traitlets.Unicode().tag(sync=True)
    y = traitlets.Unicode().tag(sync=True)
    hue = traitlets.Unicode().tag(sync=True)
    clickedValue = traitlets.Unicode().tag(sync=True)
    selectedValues = traitlets.List([]).tag(sync=True)

    def on_select_values(self, callback):
        self.observe(callback, names=["selectedValues"])

    def on_click_value(self, callback):
        self.observe(callback, names=["clickedValue"])

scatterplot = CustomScatterplot(data = affairsData.data, x = 'age', y = 'rating', hue = "children")

In [10]:
scatterplot

In [21]:
scatterplot.selectedValues

[{'rownames': 1138,
  'affairs': 12,
  'gender': 'female',
  'age': 17.5,
  'yearsmarried': 0.75,
  'children': 'yes',
  'religiousness': 2,
  'education': 12,
  'occupation': 1,
  'rating': 3,
  'id': 540}]

# ¿Cómo funciona?

En el código original tenemos las siguientes lineas de código
```js
function setValue(text) {
  model.set({ clickedValue: text });
  model.save_changes();
}

function setSelectedValues(values) {
  model.set({ selectedValues: values });
  model.save_changes();
}
```
Esto tiene como fin conectarse directamente con los atributos de nuestra clase Custom
```py
class CustomScatterplot(CustomWidget):
    _esm = CustomWidget.createWidgetFromLocalFile(paramList=["data", "x", "y", "hue"], filePath=jsPath)
    _css = cssPath
    
    data = traitlets.List([]).tag(sync=True)
    x = traitlets.Unicode().tag(sync=True)
    y = traitlets.Unicode().tag(sync=True)
    hue = traitlets.Unicode().tag(sync=True)

    # Estos nuevos atributos nos permiten interactuar con la data y regresar los datos desde el backend "javascript" hacia nuestro entorno python.
    clickedValue = traitlets.Unicode().tag(sync=True)
    selectedValues = traitlets.List([]).tag(sync=True)

    # Funciones necesarias para garantizar la reactividad. Si el valor seleccionado cambia este se guardará automáticamente en las variables especificadas
    def on_select_values(self, callback):
        self.observe(callback, names=["selectedValues"])

    def on_click_value(self, callback):
        self.observe(callback, names=["clickedValue"])
```